In [13]:
# try from script
data_config = "/home/bsb2144/daart_utils/configs/data_ibl_t.yaml"
model_config = "/home/bsb2144/daart_utils/configs/model_ibl_t.yaml"
train_config = "/home/bsb2144/daart_utils/configs/train_ibl_t.yaml"

!python ../examples/fit_models.py --data_config $data_config --model_config $model_config --train_config $train_config


DATA CONFIG:
    input_type: markers
    output_size: 5
    n_observed_classes: 5
    n_aug_classes: 0
    ignore_class: 0
    label_names: ['background', 'still', 'move', 'wheel_turn', 'groom']
    expt_ids: ['danlab_DY_009_2020-02-27-001', 'danlab_DY_018_2020-10-15-001', 'hoferlab_SWC_061_2020-11-23-001', 'mrsicflogellab_SWC_058_2020-12-11-001', 'wittenlab_ibl_witten_26_2021-01-27-002']
    expt_ids_to_keep: danlab_DY_009_2020-02-27-001;danlab_DY_018_2020-10-15-001;hoferlab_SWC_061_2020-11-23-001;mrsicflogellab_SWC_058_2020-12-11-001;wittenlab_ibl_witten_26_2021-01-27-002
    expt_ids_test: churchlandlab_CSHL045_2020-02-27-001;cortexlab_KS020_2020-02-06-001;hoferlab_SWC_043_2020-09-15-001;mrsicflogellab_SWC_052_2020-10-22-001;wittenlab_ibl_witten_27_2021-01-21-001
    data_dir: /home/bsb2144/daart_utils/data/ibl/
    results_dir: /home/bsb2144/daart/results_data/ibl/

MODEL CONFIG:
    tt_experiment_name: test_all
    model_class: segmenter
    activation: lrelu
    anneal_end_y: 12

In [ ]:
# for adding to dart need to modify
1. data gen code
2. train code

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import yaml 

from daart.testtube import get_all_params, print_hparams, create_tt_experiment, clean_tt_dir
from test_tube import HyperOptArgumentParser
from daart.data import DataGenerator, compute_sequence_pad, split_trials, SingleDataset
from daart.eval import get_precision_recall, plot_training_curves
from daart.models import Segmenter
from daart.train import Trainer
from daart.transforms import ZScore
from daart.utils import  collect_callbacks
from collections import OrderedDict
import logging
from daart.io import export_hparams
import numpy as np
import os
import pandas as pd
import pickle
import torch
from torch.utils import data
from torch.utils.data import SubsetRandomSampler
from typing import List, Union
from typeguard import typechecked

In [2]:
# check to see if the signal is markers/features
# check to see if the "idx" argument of the function is in self.batch_idxs["train"]

class IBLSingleDataset(SingleDataset):
    """Dataset class for a single dataset."""

    @typechecked
    def __getitem__(self, idx: Union[int, np.int64, None], dtype='train') -> dict:
        sample = OrderedDict()
        for signal in self.signals:

            # collect signal
            if idx is None:
                sample[signal] = [d for d in self.data[signal]]
            else:
                sample[signal] = [self.data[signal][idx]]

            # transform into tensor
            if not self.as_numpy:
                if self.dtypes[signal] == 'float32':
                    sample[signal] = torch.from_numpy(sample[signal][0]).float()
                else:
                    sample[signal] = torch.from_numpy(sample[signal][0]).long()

        # add batch index
        sample['batch_idx'] = idx

        return sample

In [3]:
class IBLDataGenerator(DataGenerator):
    _dtypes = {'train', 'val', 'test'}

    @typechecked
    def __init__(
            self,
            ids_list: List[str],
            signals_list: List[List[str]],
            transforms_list: List[list],
            paths_list: List[List[Union[str, None]]],
            device: str = 'cuda',
            as_numpy: bool = False,
            rng_seed: int = 0,
            trial_splits: Union[str, dict, None] = None,
            train_frac: float = 1.0,
            sequence_length: int = 500,
            batch_size: int = 1,
            num_workers: int = 0,
            pin_memory: bool = False,
            sequence_pad: int = 0,
            input_type: str = 'markers'
    ) -> None:

        self.ids = ids_list
        self.batch_size = batch_size
        self.as_numpy = as_numpy
        self.device = device

        self.datasets = []
        self.signals = signals_list
        self.transforms = transforms_list
        self.paths = paths_list
        for id, signals, transforms, paths in zip(
                ids_list, signals_list, transforms_list, paths_list):
            self.datasets.append(IBLSingleDataset(
                id=id, signals=signals, transforms=transforms, paths=paths, device=device,
                as_numpy=self.as_numpy, sequence_length=sequence_length,
                sequence_pad=sequence_pad, input_type=input_type))

        # collect info about datasets
        self.n_datasets = len(self.datasets)
        self.input_size = self.datasets[0].input_size
        self.feature_names = self.datasets[0].feature_names
        self.label_names = self.datasets[0].label_names

        # get train/val/test batch indices for each dataset
        if trial_splits is None:
            trial_splits = {'train_tr': 8, 'val_tr': 1, 'test_tr': 1, 'gap_tr': 0}
        elif isinstance(trial_splits, str):
            ttypes = ['train_tr', 'val_tr', 'test_tr', 'gap_tr']
            trial_splits = {
                ttype: s for ttype, s in zip(ttypes, [int(s) for s in trial_splits.split(';')])}
        else:
            pass
        self.batch_ratios = [None] * self.n_datasets
        for i, dataset in enumerate(self.datasets):
            dataset.batch_idxs = split_trials(len(dataset), rng_seed=rng_seed, **trial_splits)
            dataset.n_batches = {}
            for dtype in self._dtypes:
                if dtype == 'train':
                    # subsample training data if requested
                    if train_frac != 1.0:
                        n_batches = len(dataset.batch_idxs[dtype])
                        if train_frac < 1.0:
                            # subsample as fraction of total batches
                            n_idxs = int(np.floor(train_frac * n_batches))
                            if n_idxs <= 0:
                                print_str = (
                                    'warning: attempting to use invalid number of training '
                                    'batches; defaulting to all training batches'
                                )
                                logging.info(print_str)
                                n_idxs = n_batches
                        else:
                            # subsample fixed number of batches
                            train_frac = n_batches if train_frac > n_batches else train_frac
                            n_idxs = int(train_frac)
                        idxs_rand = np.random.choice(n_batches, size=n_idxs, replace=False)
                        dataset.batch_idxs[dtype] = dataset.batch_idxs[dtype][idxs_rand]
                    self.batch_ratios[i] = len(dataset.batch_idxs[dtype])
                dataset.n_batches[dtype] = len(dataset.batch_idxs[dtype])
        self.batch_ratios = np.array(self.batch_ratios) / np.sum(self.batch_ratios)

        # find total number of batches per data type; this will be iterated over in the train loop
        # automatically set val/test batch sizes to 1 for more fine-grained logging
        self.n_tot_batches = {}
        for dtype in self._dtypes:
            if dtype == 'train':
                self.n_tot_batches[dtype] = int(np.ceil(np.sum(
                    [dataset.n_batches[dtype] for dataset in self.datasets]) / self.batch_size))
            else:
                self.n_tot_batches[dtype] = np.sum(
                    [dataset.n_batches[dtype] for dataset in self.datasets])

        # create data loaders (will shuffle/batch/etc datasets)
        self.dataset_loaders = [None] * self.n_datasets
        for i, dataset in enumerate(self.datasets):
            self.dataset_loaders[i] = {}
            for dtype in self._dtypes:
                self.dataset_loaders[i][dtype] = torch.utils.data.DataLoader(
                    dataset,
                    batch_size=1,  # keep 1 here so we can combine batches from multiple datasets
                    sampler=SubsetRandomSampler(dataset.batch_idxs[dtype]),
                    num_workers=num_workers,
                    pin_memory=pin_memory)

        # create all iterators (will iterate through data loaders)
        self.dataset_iters = [None] * self.n_datasets
        for i in range(self.n_datasets):
            self.dataset_iters[i] = {}
            for dtype in self._dtypes:
                self.dataset_iters[i][dtype] = iter(self.dataset_loaders[i][dtype])
                
    @typechecked
    def __str__(self) -> str:
        """Pretty printing of dataset info"""
        format_str = str('Generator contains %i IBLSingleDataset objects:\n' % self.n_datasets)
        for dataset in self.datasets:
            format_str += dataset.__str__()
        return format_str
    
    @typechecked
    def next_batch(self, dtype: str, transforms: Union[iter, None]=[]) -> tuple:
        """Return next batch of data.

        The data generator iterates randomly through datasets and trials. Once a dataset runs out
        of trials it is skipped.

        Parameters
        ----------
        dtype : str
            'train' | 'val' | 'test'

        Returns
        -------
        tuple
            - sample (dict): data batch with keys given by `signals` input to class
            - dataset (int): dataset from which data batch is drawn

        """
        empty_datasets = np.zeros(self.n_datasets)

        # automatically set val/test batch sizes to 1 for more fine-grained logging
        n_batches = self.batch_size if dtype == 'train' else 1

        n_sequences = 0
        sequences = []
        datasets = []

        while True:

            # get next dataset
            dataset = np.random.choice(np.arange(self.n_datasets), p=self.batch_ratios)

            # get sequence from this dataset
            try:
                sequence = next(self.dataset_iters[dataset][dtype])
                #print('seq', sequence)
                
                #########################
                #### ADD TRANSFORMS HERE
                #########################
                # transforms on x,y paw coords for ibl data
                # (z scoreing already done)
                if dtype == 'train':
                    # do rotation transform
                    if 'rotate' in transforms:
                        angle = np.random.uniform(-10, 10)
                        #print('angle', angle)
                        temp = sequence['markers'] # shape (1, seq_len, input_len)
                        points = temp[0,:,:2]
                        points_rotated = rotate(points, degrees=angle)
                        temp[0,:,:2] = points_rotated
                        sequence['markers'] = temp
                        
                    if 'shift' in transforms:
                        u_min, u_max = -.1, .1
                        temp = sequence['markers'] # shape (1, seq_len, input_len)
                        points = temp[0,:,:2]
                        points_rotated = shift(points, u_min, u_max)
                        temp[0,:,:2] = points_rotated
                        sequence['markers'] = temp
                        
                    if 'gaussian_noise' in transforms:
                        sigma = 0.01
                        temp = sequence['markers'] # shape (1, seq_len, input_len)
                        points = temp[0,:,:]
                        points_rotated = gaussian_noise(points, sigma)
                        temp[0,:,:] = points_rotated
                        sequence['markers'] = temp
                        
                    if 'shot_noise' in transforms:
                        u_min, u_max = -1, 1
                        temp = sequence['markers'] # shape (1, seq_len, input_len)
                        points = temp[0,:,:2]
                        points_rotated = shot_noise(points, u_min, u_max)
                        temp[0,:,:2] = points_rotated
                        sequence['markers'] = temp
                       
                ########################
                ### END TRNASFORMS
                ########################
                
                # add sequence to batch
                sequences.append(sequence)
                datasets.append(dataset)
                n_sequences += 1
                # exit loop if we have enough batches
                if n_sequences == n_batches:
                    break
            except StopIteration:
                # record dataset as being empty
                empty_datasets[dataset] = 1
                # leave loop if all datasets are empty; otherwise, continue collecting sequences
                if np.all(empty_datasets):
                    break
                else:
                    continue

        batch = OrderedDict()
        if self.as_numpy:
            for i, signal in enumerate(sequences[0]):
                if signal != 'batch_idx':
                    batch[signal] = np.row_stack(
                        [s[signal].cpu().detach().numpy() for s in sequences])
                else:
                    batch['batch_idx'] = [ss['batch_idx'] for ss in sequences]
        else:
            for i, signal in enumerate(sequences[0]):
                if signal != 'batch_idx':
                    batch[signal] = torch.vstack([s[signal] for s in sequences])
                else:
                    batch['batch_idx'] = torch.vstack([s['batch_idx'] for s in sequences])

            if self.device == 'cuda':
                batch = {key: val.to('cuda') for key, val in batch.items()}

        return batch, datasets   
    

def rotate(p, origin=(0, 0), degrees=0):
    angle = np.deg2rad(degrees)
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle),  np.cos(angle)]])
    o = np.atleast_2d(origin)
    p = np.atleast_2d(p)
    return torch.tensor(np.squeeze((R @ (p.T-o.T) + o.T).T))

def shift(p, u_min, u_max):
    x_shift = np.random.uniform(u_min, u_max)
    y_shift = np.random.uniform(u_min, u_max)
    p[:,0] += x_shift
    p[:,1] += y_shift
    return p

def gaussian_noise(p, sigma):
    n = p.shape[0]
    d = p.shape[1]
    noise = np.random.normal(0, sigma, (n,d))
    p += noise
    return p

def shot_noise(p, u_min, u_max):
    ind = np.random.choice([0,1], p.shape, p=[0.99, .01])
    #print('ind', ind.shape, ind)
    noise = np.random.uniform(u_min, u_max, p.shape)
    #print('noise', noise.shape, noise)
    masked_noise = ind * noise
    #print('masked_noise',masked_noise.shape, masked_noise)
    p += masked_noise
    return p
    

In [4]:
def build_ibl_data_generator(hparams: dict, dtype='train') -> IBLDataGenerator:
    """Helper function to build a data generator from hparam dict."""

    signals = []
    transforms = []
    paths = []
    
    if dtype == 'train':
        expt_ids = hparams['expt_ids']
    else:
        expt_ids = hparams['expt_ids_test']

    for expt_id in expt_ids:

        signals_curr = []
        transforms_curr = []
        paths_curr = []

        # DLC markers or features (e.g. from simba)
        input_type = hparams.get('input_type', 'markers')
        base_dir = os.path.join(hparams['data_dir'], input_type)
        possible_markers_files = [
            os.path.join(base_dir, expt_id + '_labeled.h5'),
            os.path.join(base_dir, expt_id + '_labeled.csv'),
            os.path.join(base_dir, expt_id + '_labeled.npy'),
            os.path.join(base_dir, expt_id + '.h5'),
            os.path.join(base_dir, expt_id + '.csv'),
            os.path.join(base_dir, expt_id + '.npy'),
        ]
        markers_file = None
        for marker_file_ in possible_markers_files:
            if os.path.exists(marker_file_):
                markers_file = marker_file_
                break
        if markers_file is None:
            msg = f'did not find marker file for {expt_id} in {base_dir}'
            logging.info(msg)
            raise FileNotFoundError(msg)
        signals_curr.append('markers')
        transforms_curr.append(ZScore())
        paths_curr.append(markers_file)

        # hand labels
        if hparams.get('lambda_strong', 0) > 0:
            if expt_id not in hparams.get('expt_ids_to_keep',expt_ids) and dtype=='train':
                hand_labels_file = None
            else:
                base_dir = os.path.join(hparams['data_dir'], 'labels-hand')
                possible_hand_labels_files = [
                    os.path.join(base_dir, expt_id + '_labels.csv'),
                    os.path.join(base_dir, expt_id + '.csv'),
                ]
                hand_labels_file = None
                for hand_labels_file_ in possible_hand_labels_files:
                    if os.path.exists(hand_labels_file_):
                        hand_labels_file = hand_labels_file_
                        break
                if hand_labels_file is None:
                    logging.warning(f'did not find hand labels file for {expt_id} in {base_dir}')
            signals_curr.append('labels_strong')
            transforms_curr.append(None)
            paths_curr.append(hand_labels_file)

        # heuristic labels
        if hparams.get('lambda_weak', 0) > 0:
            base_dir = os.path.join(hparams['data_dir'], 'labels-heuristic')
            possible_heur_labels_files = [
                os.path.join(base_dir, expt_id + '_labels.csv'),
                os.path.join(base_dir, expt_id + '.csv'),
            ]
            heur_labels_file = None
            for heur_labels_file_ in possible_heur_labels_files:
                if os.path.exists(heur_labels_file_):
                    heur_labels_file = heur_labels_file_
                    break
            if heur_labels_file is None:
                logging.warning(f'did not find heuristic labels file for {expt_id} in {base_dir}')
            signals_curr.append('labels_weak')
            transforms_curr.append(None)
            paths_curr.append(heur_labels_file)

        # tasks
        if hparams.get('lambda_task', 0) > 0:
            tasks_labels_file = os.path.join(hparams['data_dir'], 'tasks', expt_id + '.csv')
            signals_curr.append('tasks')
            transforms_curr.append(ZScore())
            paths_curr.append(tasks_labels_file)

        # define data generator signals
        signals.append(signals_curr)
        transforms.append(transforms_curr)
        paths.append(paths_curr)

    # compute padding needed to account for convolutions
    hparams['sequence_pad'] = compute_sequence_pad(hparams)

    # build data generator
    ibl_data_gen = IBLDataGenerator(
        expt_ids, signals, transforms, paths,
        device=hparams['device'],
        sequence_length=hparams['sequence_length'],
        sequence_pad=hparams['sequence_pad'],
        batch_size=hparams['batch_size'],
        trial_splits=hparams['trial_splits'],
        train_frac=hparams['train_frac'],
        input_type=hparams.get('input_type', 'markers'),
    )

    # automatically compute input/output sizes from data
    hparams['input_size'] = ibl_data_gen.input_size
    hparams['output_size'] = len(ibl_data_gen.label_names)

    if hparams.get('lambda_task', 0) > 0:
        task_size = 0
        for batch in ibl_data_gen.datasets[0].data['tasks']:
            if batch.shape[1] == 0:
                continue
            else:
                task_size = batch.shape[1]
                break
        hparams['task_size'] = task_size

    return ibl_data_gen

In [5]:
# set config paths
data_config = "/home/bsb2144/daart_utils/configs/data_ibl_t.yaml"
model_config = "/home/bsb2144/daart_utils/configs/model_ibl_t.yaml"
train_config = "/home/bsb2144/daart_utils/configs/train_ibl_t.yaml"

hparams = {}
#namespace, extra = parser.parse_known_args()

# add arguments from all configs
configs = [data_config, model_config, train_config]
for config in configs:
    config_dict = yaml.safe_load(open(config))
    for (key, value) in config_dict.items():
        hparams[key] = value
        
create_tt_experiment(hparams)
print('done')

done


In [6]:
# -------------------------------------
# build data generator
# -------------------------------------
import random
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)
data_gen = build_ibl_data_generator(hparams)
#print(data_gen)

# pull class weights out of labeled training data
if hparams.get('weight_classes', False):
    totals = data_gen.count_class_examples()
    idx_background = hparams.get('ignore_class', 0)
    if idx_background in np.arange(len(totals)):
        totals[idx_background] = 0  # get rid of background class
    # select class weights by choosing class with max labeled examples to have a value of 1;
    # the remaining weights will be inversely proportional to their prevalence. For example, a
    # class that has half as many examples as the most prevalent will be weighted twice as much
    class_weights = np.max(totals) / (totals + 1e-10)
    class_weights[totals == 0] = 0
    hparams['class_weights'] = class_weights.tolist()  # needs to be list to save out to yaml
    #print('class weights: {}'.format(class_weights))
else:
    hparams['class_weights'] = None

In [7]:
# see what data generator returns
import random
torch.manual_seed(0)
random.seed(0)
np.random.seed(0)
transforms =['shot_noise']
data, dataset = data_gen.next_batch('train', transforms=transforms)
print(data.keys())
print()

# batch index per sequence
print(data['batch_idx'])

# shape (n_sequences, sequence_length, n_markers)
print(data['markers'].shape)

# shape (n_sequences, sequence_length)
print(data['labels_strong'].shape)

# show one batch
print(data['markers'][0][:10])
print(torch.sum(data['markers'][0]))

dict_keys(['markers', 'labels_strong', 'batch_idx'])

tensor([[14],
        [89],
        [82],
        [12],
        [53],
        [12],
        [48],
        [20]], device='cuda:0')
torch.Size([8, 1048, 3])
torch.Size([8, 1048])
tensor([[ 0.5238, -0.4628,  0.9411],
        [ 0.4514, -0.3292,  0.8752],
        [ 0.3616, -0.2738,  0.8339],
        [ 0.3272, -0.1445,  0.8150],
        [ 0.3904,  0.0141,  0.7965],
        [ 0.5189,  0.1581,  0.7253],
        [ 0.6257,  0.3257,  0.5347],
        [ 0.6961,  0.4609,  0.1866],
        [ 0.6891,  0.4971, -0.2856],
        [ 0.6750,  0.3975, -0.7801]], device='cuda:0')
tensor(104.2548, device='cuda:0')


In [8]:
# # as is 
# tensor([[ 0.5238, -0.4628,  0.9411],
#         [ 0.4514, -0.3292,  0.8752],
#         [ 0.3616, -0.2738,  0.8339],
#         [ 0.3272, -0.1445,  0.8150],
#         [ 0.3904,  0.0141,  0.7965],
#         [ 0.5189,  0.1581,  0.7253],
#         [ 0.6257,  0.3257,  0.5347],
#         [ 0.6961,  0.4609,  0.1866],
#         [ 0.6891,  0.4971, -0.2856],
#         [ 0.6750,  0.3975, -0.7801]], device='cuda:0')
# summ = tensor(106.6321, device='cuda:0')
# # shot
# tensor([[ 0.5238, -0.4628,  0.9411],
#         [ 0.4514, -0.3292,  0.8752],
#         [ 0.3616, -0.2738,  0.8339],
#         [ 0.3272, -0.1445,  0.8150],
#         [ 0.3904,  0.0141,  0.7965],
#         [ 0.5189,  0.1581,  0.7253],
#         [ 0.6257,  0.3257,  0.5347],
#         [ 0.6961,  0.4609,  0.1866],
#         [ 0.6891,  0.4971, -0.2856],
#         [ 0.6750,  0.3975, -0.7801]], device='cuda:0')





# # 

NameError: name 'tensor' is not defined

In [9]:
# -------------------------------------
# build model
# -------------------------------------
model = Segmenter(hparams)
model.to(device='cuda')
print(model)


DTCN architecture
------------------------
Encoder:
    0: DilationBlock
        0: Conv1d(3, 32, kernel_size=(9,), stride=(1,), padding=(4,))
        1: LeakyReLU(negative_slope=0.05)
        2: Dropout2d(p=0.1, inplace=False)
        3: Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(4,))
        4: LeakyReLU(negative_slope=0.05)
        5: Dropout2d(p=0.1, inplace=False)
        6: residual connection
        7: LeakyReLU(negative_slope=0.05)

    1: DilationBlock
        0: Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(8,), dilation=(2,))
        1: LeakyReLU(negative_slope=0.05)
        2: Dropout2d(p=0.1, inplace=False)
        3: Conv1d(32, 32, kernel_size=(9,), stride=(1,), padding=(8,), dilation=(2,))
        4: LeakyReLU(negative_slope=0.05)
        5: Dropout2d(p=0.1, inplace=False)
        6: residual connection
        7: LeakyReLU(negative_slope=0.05)


Classifier:
    0: Linear(in_features=32, out_features=5, bias=True)




In [10]:
# -------------------------------------
# train model
# -------------------------------------
callbacks = collect_callbacks(hparams)
trainer = Trainer(**hparams, callbacks=callbacks)
trainer.fit(model, data_gen, save_path=hparams['tt_version_dir'])

# update hparams upon successful training
hparams['training_completed'] = True
export_hparams(hparams)

  0%|          | 0/502 [00:00<?, ?it/s]/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "
100%|██████████| 502/502 [06:26<00:00,  1.30it/s]


In [11]:
# -------------------------------------
# export artifacts
# -------------------------------------

# save training curves
if hparams.get('plot_train_curves', False):
    plot_training_curves(
        metrics_file=os.path.join(hparams['tt_version_dir'], 'metrics.csv'),
        dtype='train',
        expt_ids=hparams['expt_ids'],
        save_file=os.path.join(hparams['tt_version_dir'], 'train_curves'),
        format='png',
    )
    # plot_training_curves(
    #     metrics_file=os.path.join(hparams['tt_version_dir'], 'metrics.csv'),
    #     dtype='val',
    #     expt_ids=hparams['expt_ids'],
    #     save_file=os.path.join(hparams['tt_version_dir'], 'val_curves'),
    #     format='png',
    # )

# run model inference on all training sessions
if hparams['train_frac'] != 1.0:  # rebuild data generator to include all data if necessary
    hparams['train_frac'] = 1.0
    data_gen = build_ibl_data_generator(hparams)
results_dict = model.predict_labels(data_gen)
for sess, dataset in enumerate(data_gen.datasets):
    expt_id = dataset.id
    labels = np.vstack(results_dict['labels'][sess])
    np.save(os.path.join(hparams['tt_version_dir'], f'{expt_id}_states.npy'), labels)
    
# run model inference on all test sessions
data_gen = build_ibl_data_generator(hparams, dtype='test')
results_dict = model.predict_labels(data_gen)
for sess, dataset in enumerate(data_gen.datasets):
    expt_id = dataset.id
    labels = np.vstack(results_dict['labels'][sess])
    np.save(os.path.join(hparams['tt_version_dir'], f'{expt_id}_states.npy'), labels)

/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


In [12]:

# get rid of unneeded logging info
clean_tt_dir(hparams)